In [1]:
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm
tqdm.pandas()
import numpy as np
import pandas as pd
import re
import os

RDLogger.DisableLog("rdApp.*")

/Users/mariacm/miniforge3/envs/openadmet-models/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Additional curation for CYP targets

`curate_data_IC50.ipynb` deposits ChEMBL IC50 data permissively (i.e., every record with a pChEMBL value). That is the right baseline, but CYP inhibition data carries a few
failure modes that make the permissive set unsuitable for model training as it stands:

- **time-dependent inhibition (TDI)** measured alongside reversible IC50s, which is a different endpoint
- **non-human orthologues** mixed in with the human enzyme
- records whose description names a **different isoform** entirely
- **censoring thresholds stored as exact values** ("no inhibition at 30 uM" deposited as an IC50)
- a **unit error** in one patent, where uM values were transcribed as nM

This notebook reads the deposition produced by `curate_data_IC50.ipynb`, applies those filters, and
writes a CYP-specific deposition in the same format. TDI records are not discarded, but rather kept
as their own set.

It also records two things the permissive deposition does not:

- **`substrate`** — the probe substrate, recovered from the free-text assay description. Values
  measured against different probes are not interchangeable, so aggregation groups on it.
- **`CHEMBL_INCHIKEY`** — the InChIKey of the structure ChEMBL actually deposited. The pipeline's
  `canonical_smiles()` runs a tautomer canonicalisation that returns a *different molecule* for
  ~1.6% of compounds (nootkatone, nifedipine, tenidap...), so `OPENADMET_INCHIKEY` cannot be used
  to join to PubChem or any external source for those. 

## Define target metadata

In [2]:
targets = {
    "CYP1A2": "CHEMBL3356",
    "CYP2B6": "CHEMBL4729",
    "CYP2C8": "CHEMBL3721",
    "CYP2C9": "CHEMBL3397",
    "CYP2C19": "CHEMBL3622",
    "CYP2D6": "CHEMBL289",
    "CYP3A4": "CHEMBL340",
}

In [3]:
chembl_ver = 37

In [4]:
source_location = f"ChEMBL{chembl_ver}_IC50"
location = f"ChEMBL{chembl_ver}_IC50_curated"

## Probe substrate

ChEMBL has no substrate field; the probe is named only inside `assay_description`. Hits are
restricted to probes that are canonical for the isoform in question, so that a multi-CYP panel
description does not attribute another isoform's probe to this one.

In [5]:
SUBSTRATE_PATTERNS = {
    "phenacetin": r"phenacetin",
    "ethoxyresorufin": r"ethoxy-?resorufin|\bEROD\b|7-ethyl-0-resorufin",
    "methoxyresorufin": r"methoxyresorufin",
    "benzyloxyresorufin": r"benzyloxyresorufin|\bBzRes\b",
    "caffeine": r"caffeine",
    "tacrine": r"tacrine",
    "CEC": r"3-cyano-7-ethoxycoumarin|\bCEC\b|cyanoethoxycoumarin",
    "diclofenac": r"diclofenac",
    "tolbutamide": r"tolbutamide",
    "warfarin": r"warfarin",
    "flurbiprofen": r"flurbiprofen",
    "MFC": r"7-methoxy-4-\(?trifluoromethyl\)?coumarin|\bMFC\b",
    "dextromethorphan": r"dextromethorphan",
    "bufuralol": r"bufuralol",
    "debrisoquine": r"debrisoquine",
    "AMMC": r"\bAMMC\b",
    "MAMC": r"\bMAMC\b|7-methoxy-4-\(?aminomethyl\)?-?coumarin",
    "midazolam": r"midazolam",
    "testosterone": r"testosterone",
    "nifedipine": r"nifedipine",
    "felodipine": r"felodipine",
    "erythromycin": r"erythromycin",
    "atorvastatin": r"atorvastatin",
    "simvastatin": r"simvastatin",
    "BFC": r"7-benzyloxy-?4-?\(?trifluoromethyl\)?\s?coumarin|\bBFC\b",
    "7-benzyloxyquinoline": r"benzyloxyquinoline|\b7-BQ\b",
    "DBF": r"dibenzylfluorescein|\bDBF\b",
    "diethoxyfluorescein": r"diethoxyfluorescein|\bDEF\b",
    "bupropion": r"bupropion",
    "EFC": r"7-ethoxy-4-\(?trifluoromethyl\)?coumarin|\bEFC\b",
    "S-mephenytoin": r"mephenytoin|mpheytoin",
    "omeprazole": r"omeprazole",
    "OMF": r"3-O-methylfluorescein|\bOMF\b",
    "paclitaxel": r"paclitaxel|taxol",
    "amodiaquine": r"amodiaquin",
    "luciferin": r"luciferin|P450-Glo",
}
SUBSTRATE_RE = {k: re.compile(v, re.I) for k, v in SUBSTRATE_PATTERNS.items()}

#  The substrates typically used for each isoform, as reported in the literature.
#  "luciferin" is the generic P450-Glo probe and is valid for every isoform
CANONICAL_SUBSTRATES = {
    "CYP1A2": {"phenacetin", "ethoxyresorufin", "methoxyresorufin", "caffeine", "tacrine", "CEC"},
    "CYP2B6": {"bupropion", "EFC"},
    "CYP2C8": {"paclitaxel", "amodiaquine", "DBF"},
    "CYP2C9": {"diclofenac", "tolbutamide", "warfarin", "flurbiprofen", "MFC", "DBF", "CEC"},
    "CYP2C19": {"S-mephenytoin", "omeprazole", "CEC", "OMF"},
    "CYP2D6": {"dextromethorphan", "bufuralol", "debrisoquine", "AMMC", "MAMC"},
    "CYP3A4": {"midazolam", "testosterone", "nifedipine", "felodipine", "erythromycin", "atorvastatin",
               "simvastatin", "BFC", "7-benzyloxyquinoline", "DBF", "diethoxyfluorescein",
               "benzyloxyresorufin"},
}

def find_substrate(desc, target):
    """The probe substrate named in the description.

    Panel descriptions list every isoform's probe, so they cannot say which one this measurement
    used; those are reported as "not stated" rather than guessed at.
    """
    d = str(desc)
    allowed = CANONICAL_SUBSTRATES.get(target, set()) | {"luciferin"}
    hits = sorted(k for k, pat in SUBSTRATE_RE.items() if k in allowed and pat.search(d))
    return hits[0] if len(hits) == 1 else "not stated"

## Curation rules

One predicate per reason a record is dropped.

In [6]:
# Explicit TDI vocabulary.
TDI_NAMED = (r"time[- ]dependent|mechanism[- ]based|metabolis[mt][- ]dependent"
             r"|IC50 shift|fold[- ]shift|irreversible|\bTDI\b|\bMDI\b")
# A TDI protocol even when the description never says "time-dependent": NADPH is present DURING the
# pre-incubation, so the compound is metabolised before the probe substrate goes in.
TDI_PROTOCOL = (r"pre-?incubat\w*[^.]{0,60}?(?:with|in (?:the )?presence of)[^.]{0,30}NADPH"
                r"|NADPH[^.]{0,40}pre-?incubat")
# ...unlike an ordinary reversible IC50, where NADPH is what STARTS the reaction afterwards.
REVERSIBLE = (r"reactions? (?:are|is|was) started with the cofactor|the reaction was started"
              r"|pre-?incubat[^.]{0,60}(?:followed(?: by)?|then|prior to)[^.]{0,30}NADPH "
              r"(?:addition|regenerating|was added)")

ISO = r"(?:CYP|P450|P-450)\s*-?\s*(\d[A-Z]\d{1,2})"

# ChEMBL records patent US-9447092-B2 as 30 nM for CYP2C9/CYP2C19/CYP2D6 but as 30000 nM (= 30 uM)
# for CYP1A2, so the uM values were transcribed as nM for the  other three isoforms. 
UNIT_ERROR_PATENT = "US-9447092-B2"
UNIT_ERROR_MAX_NM = 1000

# An assay where EVERY record carries the identical value is a censoring threshold stored as an exact IC50. 
# ChEMBL only assigns a pChEMBL value to exact (=) records, so these arrive looking like genuine data.
MIN_CONSTANT_ROWS = 4


def is_tdi(desc):
    """Time-dependent / metabolism-dependent inhibition, named or inferred from the protocol."""
    d = str(desc)
    if re.search(TDI_NAMED, d, re.I):
        return True
    return bool(re.search(TDI_PROTOCOL, d, re.I)) and not re.search(REVERSIBLE, d, re.I)


def other_isoform(desc, target):
    """Short descriptions that name other isoforms but never the target.

    Long ones are usually truncated multi-CYP panels that do mention it further in.
    """
    named = {m.upper() for m in re.findall(ISO, str(desc), re.I)}
    return bool(named) and target[3:] not in named and len(str(desc)) < 200


def constant_value_assays(r, min_rows=MIN_CONSTANT_ROWS):
    """assay_ids whose every record carries the same standard_value."""
    g = r.groupby("assay_id")["standard_value"].agg(["size", "nunique"])
    return g.index[(g["size"] >= min_rows) & (g["nunique"] == 1)]


def removal_reasons(r, target):
    """One boolean column per reason a raw record is dropped."""
    return pd.DataFrame({
        # rows with no organism annotated are kept: they are '(unknown origin)' assays on the human target
        "non_human": r["assay_organism"].notna() & (r["assay_organism"] != "Homo sapiens"),
        "other_isoform": r["assay_description"].apply(other_isoform, target=target),
        "tdi": r["assay_description"].apply(is_tdi),
        "unit_error": ((r["doc_patent_id"].astype("string") == UNIT_ERROR_PATENT)
                       & (r["standard_value"] < UNIT_ERROR_MAX_NM)),
        "constant_value_assay": r["assay_id"].isin(constant_value_assays(r)),
    }, index=r.index)

## Curate a target

Returns the reversible set, the TDI set, and a row for the summary table. Aggregation groups on
`substrate` as well as compound, so a compound measured against two probes gives two rows.

In [7]:
def inchikey_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    return Chem.MolToInchiKey(mol) if mol is not None else pd.NA


def aggregate_by_compound(r, by_substrate=True):
    """Same aggregation as the permissive deposition, optionally split by probe substrate."""
    keys = ["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY"] + (["substrate"] if by_substrate else [])
    data = r.groupby(keys).agg({"assay_id": "count",
                                "standard_value": ["mean", "median", "std"],
                                "pchembl_value": ["mean", "median", "std"]})
    data.columns = ["_".join(col) for col in data.columns]
    return data.reset_index().sort_values("assay_id_count", ascending=False)


def curate_cyp_target(target_name: str, chembl_tid: str, source_path):
    print(f"working on target {target_name}")
    fname = f"ChEMBL_IC50_{target_name}_{chembl_tid}_raw.parquet"
    raw = pd.read_parquet(source_path / fname)
    print(f"  permissive records: {len(raw)}")

    # the structure ChEMBL deposited, which OPENADMET_INCHIKEY does not always represent
    keys = {s: inchikey_from_smiles(s) for s in raw["canonical_smiles"].dropna().unique()}
    raw["CHEMBL_INCHIKEY"] = raw["canonical_smiles"].map(keys)
    raw["substrate"] = raw["assay_description"].apply(find_substrate, target=target_name)

    reasons = removal_reasons(raw, target_name)
    keep = ~reasons.any(axis=1)
    clean = raw[keep].drop(columns=["tdi", "other_isoform"], errors="ignore").reset_index(drop=True)
    # TDI records are an endpoint of their own, so they are kept rather than discarded -- but only
    # those that pass every other check
    tdi = raw[reasons["tdi"] & ~reasons.drop(columns="tdi").any(axis=1)].reset_index(drop=True)

    summary = {**reasons.sum().to_dict(),
               "raw_before": len(raw), "raw_after": int(keep.sum()),
               "compounds_after": clean["OPENADMET_INCHIKEY"].nunique(),
               "tdi_records": len(tdi),
               "key_differs": int(raw.drop_duplicates("OPENADMET_INCHIKEY")
                                  .eval("CHEMBL_INCHIKEY != OPENADMET_INCHIKEY").sum())}
    print(f"  dropped {int((~keep).sum())} "
          f"({', '.join(f'{k} {int(v)}' for k, v in reasons.sum().items() if v)})")
    print(f"  kept {len(clean)} reversible records, {len(tdi)} TDI records")
    return clean, aggregate_by_compound(clean), tdi, aggregate_by_compound(tdi) if len(tdi) else None, summary

# Setup S3

`PUSH_TO_S3` defaults to `False`: rerunning this notebook regenerates the local parquet files but
does not touch S3 or the catalog until someone has reviewed the output. The catalog is written to
its own `location`, so it cannot overwrite the permissive deposition's catalog.

In [8]:
PUSH_TO_S3 = False
bucket_name = "openadmet-data-public-dev"

In [9]:
from pathlib import Path

source_path = Path(source_location)
location_path = Path(location)
location_path.mkdir(exist_ok=True)

In [10]:
if PUSH_TO_S3:
    from openadmet.toolkit.webservices.credentials import S3Settings
    from openadmet.toolkit.webservices.s3 import S3Bucket

    assert "AWS_ACCESS_KEY_ID" in os.environ and "AWS_SECRET_ACCESS_KEY" in os.environ, (
        "set AWS credentials in the environment before running this cell "
        "(never shell out to `aws configure get` in a committed notebook)"
    )
    settings = S3Settings()
    bucket = S3Bucket.from_settings(settings, bucket_name)

# Main loop

In [11]:
uris_raw = {}
uris_agg = {}
summaries = {}

for target, chembl_tid in targets.items():
    clean, agg, tdi, tdi_agg, summary = curate_cyp_target(target, chembl_tid, source_path)
    summaries[target] = summary

    outputs = {
        f"ChEMBL_IC50_CYP_{target}_{chembl_tid}_aggregated.parquet": (agg, uris_agg, target + "_aggregated"),
        f"ChEMBL_IC50_CYP_{target}_{chembl_tid}_raw.parquet": (clean, uris_raw, target + "_raw"),
    }
    if tdi_agg is not None:
        outputs[f"ChEMBL_IC50_CYP_TDI_{target}_{chembl_tid}_aggregated.parquet"] = (
            tdi_agg, uris_agg, target + "_TDI_aggregated")
        outputs[f"ChEMBL_IC50_CYP_TDI_{target}_{chembl_tid}_raw.parquet"] = (
            tdi, uris_raw, target + "_TDI_raw")

    for fname, (df, uris, entry) in outputs.items():
        df.reset_index(drop=True).to_parquet(location_path / fname, index=False)
        if PUSH_TO_S3:
            destination = location + "/" + fname
            bucket.push_file(location_path / fname, destination)
            uris[entry] = bucket.to_uri(destination)

print(f"\nwrote {len(list(location_path.glob('*.parquet')))} parquet files to {location_path}/")

working on target CYP1A2
  permissive records: 1764
  dropped 37 (non_human 2, tdi 8, constant_value_assay 27)
  kept 1727 reversible records, 8 TDI records
working on target CYP2B6
  permissive records: 240
  dropped 7 (other_isoform 5, tdi 2)
  kept 233 reversible records, 2 TDI records
working on target CYP2C8
  permissive records: 493
  dropped 10 (other_isoform 1, tdi 9)
  kept 483 reversible records, 9 TDI records
working on target CYP2C9
  permissive records: 3107
  dropped 88 (non_human 3, other_isoform 2, tdi 10, unit_error 5, constant_value_assay 73)
  kept 3019 reversible records, 10 TDI records
working on target CYP2C19
  permissive records: 1735
  dropped 63 (tdi 3, unit_error 4, constant_value_assay 60)
  kept 1672 reversible records, 3 TDI records
working on target CYP2D6
  permissive records: 3348
  dropped 49 (other_isoform 1, tdi 27, unit_error 10, constant_value_assay 11)
  kept 3299 reversible records, 27 TDI records
working on target CYP3A4
  permissive records: 66

In [12]:
summary_table = pd.DataFrame(summaries).T
summary_table["% kept"] = (100 * summary_table["raw_after"] / summary_table["raw_before"]).round(1)
summary_table.to_csv(location_path / f"curation_summary_{location}.csv")
summary_table

,non_human,other_isoform,tdi,unit_error,constant_value_assay,raw_before,raw_after,compounds_after,tdi_records,key_differs,% kept
CYP1A2,2.0,0.0,8.0,0.0,27.0,1764.0,1727.0,1536.0,8.0,14.0,97.9
CYP2B6,0.0,5.0,2.0,0.0,0.0,240.0,233.0,200.0,2.0,2.0,97.1
CYP2C8,0.0,1.0,9.0,0.0,0.0,493.0,483.0,453.0,9.0,16.0,98.0
CYP2C9,3.0,2.0,10.0,5.0,73.0,3107.0,3019.0,2740.0,10.0,73.0,97.2
CYP2C19,0.0,0.0,3.0,4.0,60.0,1735.0,1672.0,1518.0,3.0,20.0,96.4
CYP2D6,0.0,1.0,27.0,10.0,11.0,3348.0,3299.0,2920.0,27.0,24.0,98.5
CYP3A4,13.0,3.0,220.0,0.0,134.0,6642.0,6272.0,5231.0,220.0,83.0,94.4


# Build the Intake Catalog

Guarded by `PUSH_TO_S3`, and written under this deposition's own `location` so the permissive
catalog is untouched.

In [13]:
if PUSH_TO_S3:
    import intake

    cat = intake.entry.Catalog()
    for k, v in uris_agg.items():
        cat[k] = intake.readers.PandasParquet(v)
    for k, v in uris_raw.items():
        cat[k] = intake.readers.PandasParquet(v)

    catname = f"CATALOG_{location}.yaml"
    cat.to_yaml_file(catname)
    cat_location = location + "/" + catname
    bucket.push_file(catname, cat_location)
    print(f"pushed catalog to {cat_location} with {len(uris_agg) + len(uris_raw)} entries")
else:
    print("PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built")

PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built
